In [ ]:
import os
import sys
import psutil
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
from joblib.externals.loky import get_reusable_executor
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "../support/")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() - 1 or 1
_TOTAL_MEMORY = psutil.virtual_memory().total
_AVAILABLE_MEMORY = psutil.virtual_memory().available
_MEMORY_PER_WORKER = max(100 * 1024**2, _AVAILABLE_MEMORY // (_NCPU + 1))
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | {_TOTAL_MEMORY / 1024**3:.1f}GB total RAM ({_AVAILABLE_MEMORY / 1024**3:.1f}GB available) | pyarrow {pa.__version__}", flush=True)


Running with 47 CPU cores | 92.3GB total RAM (78.0GB available) | pyarrow 24.0.0


Variables

In [2]:
%run ../0_Config/0_variables.ipynb

Feature selection

In [3]:
import time as _time

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_fdu_path = resolve(f"3_Features_select/Selected_features/{_target}_feature_data_unique_{_stem}.parquet")
print(f"Loading unique feature_data parquet: {_fdu_path}", flush=True)
_t = _time.perf_counter()
feature_data_unique = pd.read_parquet(_fdu_path)
print(f"  loaded feature_data_unique: shape={feature_data_unique.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

print(f"Loading features parquet: {os.environ['FEATURES_PATH']}", flush=True)
_t = _time.perf_counter()
features = pd.read_parquet(os.environ["FEATURES_PATH"], filters=[
    ('SETTLEMENTDATE', '>=', pd.Timestamp(os.environ["FEATURE_DATASET_START"])),
    ('SETTLEMENTDATE', '<=', pd.Timestamp(os.environ["FEATIRE_DATASET_END"])),
])
features = features.drop(columns=[c for c in features.columns if c in set(os.environ["TARGET_COLS"].split(","))])
features = features.loc[:pd.Timestamp(os.environ["FEATIRE_DATASET_END"])]
print(f"  loaded features: shape={features.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_tagg_path = resolve(f"3_Features_select/Selected_features/{_target}_targets_agg_{_stem}.parquet")
print(f"Loading aggregated targets parquet: {_tagg_path}", flush=True)
_t = _time.perf_counter()
future_prediction_targets_agg = pd.read_parquet(_tagg_path)
print(f"  loaded targets_agg: shape={future_prediction_targets_agg.shape} in {_time.perf_counter() - _t:.1f}s", flush=True)

def feature_data_selection(
    feature_data,
    features,
    future_prediction_targets_agg,
    subsample_start: pd.Timestamp,
    subsample_end: pd.Timestamp,
    subsample_amount: int,
):
    # Number of equally-spaced k values to evaluate between K_MIN and K_MAX — tune this
    N_K_VALUES = 10
    # Optional bounds for the k sweep (inclusive). None = use 1 / n_available_features respectively.
    K_MIN = 10
    K_MAX = None
    # Number of equally-spaced horizons to evaluate — e.g. 10 tests every ~10th horizon; None = all
    N_HORIZON_STEPS = None
    N_CV_SPLITS = 3   # time-series walk-forward folds
    # Number of independent subsample / LightGBM RNG seeds. Each seed re-draws the row subsample
    # and re-seeds LightGBM bagging; per-(h,k) MAE is averaged over (N_SUBSAMPLE_SEEDS × N_CV_SPLITS)
    # samples before picking best_k. Cheap insurance against picking a noisy k when adjacent
    # grid points differ by < within-fit noise. Set to 1 to recover the original single-seed run.
    N_SUBSAMPLE_SEEDS = 5
    _SUBSAMPLE_SEEDS = [42 + i for i in range(N_SUBSAMPLE_SEEDS)]
    # Outer thread count — each task sweeps ALL folds × k for one horizon (per-horizon task layout).
    # Real values set below once n_tasks is known; placeholder here so _LGBM_CV_PARAMS can reference it.
    N_PARALLEL_JOBS = _NCPU
    _LGBM_INNER_THREADS = 1

    # Lightweight LightGBM for CV speed — not production params.
    # Tuned aggressively for ranking-of-k (not absolute MAE accuracy):
    #   - num_leaves=31: 2× capacity vs 15 so CV can distinguish feature-set sizes;
    #   - learning_rate=0.1 + n_estimators=200 (fixed): no eval_set / no early stopping (#10) avoids
    #     per-iteration validation cost (~15-25% faster). Fixed 100 trees converges fine at lr=0.1.
    #   - bagging_fraction=0.5, bagging_freq=1: ~2× faster per tree iteration; adds noise but k-ranking
    #     is robust to it.
    #   - force_col_wise=True: skips LightGBM's layout auto-detection, goes straight to column-wise
    #     (matches our Fortran-ordered X_arr, see below).
    _LGBM_CV_PARAMS = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": 200,            # fixed; no early stopping
        "learning_rate": 0.1,
        "num_leaves": 31,
        "min_child_samples": 20,
        "bagging_fraction": 0.5,
        "bagging_freq": 1,
        "force_col_wise": True,
        "random_state": 42,
        "n_jobs": _LGBM_INNER_THREADS,      # sklearn-style alias
        "num_threads": _LGBM_INNER_THREADS, # canonical name used by lgb.train (some builds ignore n_jobs here)
        "verbose": -1,
    }

    horizon_cols = [c for c in feature_data.columns if c.startswith("h") and c[1:].isdigit()]
    mi_matrix = feature_data.set_index("feature")[horizon_cols]  # (n_features, n_horizons)
    available_features = [f for f in mi_matrix.index if f in features.columns]  # Only keep features that exist in the in-memory features DataFrame
    print(f"  available features: {len(available_features)} of {len(mi_matrix)} (intersection with loaded features parquet)", flush=True)

    # Accumulator across all seeds and folds — keyed by (h_idx, k), value is list of MAEs.
    fold_mae_acc = {}

    for _seed_idx, _seed in enumerate(_SUBSAMPLE_SEEDS):
        print(
            f"\n=== seed {_seed_idx + 1}/{len(_SUBSAMPLE_SEEDS)} (rng={_seed}) ===",
            flush=True,
        )

        # Time-range filter then random row subsample (varies per seed)
        shared_idx = features.index.intersection(future_prediction_targets_agg.index)
        shared_idx = shared_idx[(shared_idx >= subsample_start) & (shared_idx <= subsample_end)]
        n_total = len(shared_idx)
        n_rows = min(subsample_amount, n_total)
        if n_rows < n_total:
            rng = np.random.default_rng(_seed)
            chosen = np.sort(rng.choice(n_total, size=n_rows, replace=False))
            shared_idx = shared_idx[chosen]
        print(f"CV subsample: {n_rows:,} rows (of {n_total:,} in range {subsample_start.date()} – {subsample_end.date()})", flush=True)

        # Arrays live in the main process; threads share them without copying (no pickling, no mmap needed)
        print(f"  materialising X_arr / Y_arr / mi_arr as float32 ndarrays", flush=True)
        _t = _time.perf_counter()
        # to_numpy(copy=False) avoids a redundant cast when underlying blocks are already float32.
        X_arr = features.loc[shared_idx, available_features].to_numpy(dtype=np.float32, copy=False)  # (n_rows, n_features)
        # Fortran (column-major) layout: LightGBM consumes data column-wise; F-order eliminates the
        # internal transpose that happens on every fit() call with C-order input. ~10-20% faster fits.
        if not X_arr.flags["F_CONTIGUOUS"]:
            X_arr = np.asfortranarray(X_arr)
        Y_arr = future_prediction_targets_agg.loc[shared_idx].to_numpy(dtype=np.float32, copy=False)  # (n_rows, n_horizons)
        if not Y_arr.flags["C_CONTIGUOUS"]:
            Y_arr = np.ascontiguousarray(Y_arr)
        mi_arr = mi_matrix.loc[available_features].to_numpy(dtype=np.float32, copy=False)             # (n_features, n_horizons)
        if not mi_arr.flags["C_CONTIGUOUS"]:
            mi_arr = np.ascontiguousarray(mi_arr)
        print(
            f"    arrays ready in {_time.perf_counter() - _t:.1f}s | "
            f"X={X_arr.nbytes / 1024**2:.0f}MB Y={Y_arr.nbytes / 1024**2:.0f}MB mi={mi_arr.nbytes / 1024**2:.0f}MB",
            flush=True,
        )

        x_mb = X_arr.nbytes / 1e6
        print(f"X_arr: {X_arr.shape} ({x_mb:.0f} MB) | threads: {N_PARALLEL_JOBS}", flush=True)

        tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
        fold_indices = list(tscv.split(X_arr))  # pre-compute once

        # Equally-spaced grid from K_MIN → K_MAX (defaulting to 1 → n_available_features), deduplicated and sorted.
        # Bounds are clipped to [1, n_available_features] so out-of-range values just collapse to the edge.
        _k_lo = 1 if K_MIN is None else max(1, int(K_MIN))
        _k_hi = len(available_features) if K_MAX is None else min(len(available_features), int(K_MAX))
        if _k_lo > _k_hi:
            _k_lo, _k_hi = _k_hi, _k_lo
        k_grid = sorted(set(int(round(k)) for k in np.logspace(np.log10(max(1, _k_lo)), np.log10(max(1, _k_hi)), N_K_VALUES)))

        # Equally-spaced horizon indices to evaluate; None = all horizons
        n_h = len(horizon_cols)
        if N_HORIZON_STEPS is None or N_HORIZON_STEPS >= n_h:
            horizon_indices = list(range(n_h))
        else:
            horizon_indices = sorted(set(int(round(i)) for i in np.linspace(0, n_h - 1, N_HORIZON_STEPS)))

        n_tasks = len(horizon_indices)  # one task per horizon — folds × k looped inside
        # Refine outer/inner thread split now that we know the real task count.
        # Memory cap: each worker materialises one (n_rows, n_features) F-order copy of X_sorted
        # plus per-fold slices and LightGBM Dataset bins → budget ~8× X_arr per worker, then
        # leave ≥25% of system RAM free for the parent's X_arr/Y_arr/mi_arr and OS cache.
        _per_worker_bytes = max(1, int(X_arr.nbytes * 8))
        _ram_budget = max(1, int(_AVAILABLE_MEMORY * 0.75))
        _mem_cap = max(1, _ram_budget // _per_worker_bytes)
        N_PARALLEL_JOBS = max(1, min(_NCPU, n_tasks, _mem_cap))
        _LGBM_INNER_THREADS = max(1, _NCPU // N_PARALLEL_JOBS)
        print(
            f"  worker mem budget: {_per_worker_bytes / 1024**2:.0f}MB/worker, "
            f"{_ram_budget / 1024**3:.1f}GB available → mem-cap={_mem_cap} workers",
            flush=True,
        )
        # Re-seed LightGBM per outer seed so bagging RNG also varies across seed iterations.
        _LGBM_CV_PARAMS["random_state"] = int(_seed)
        _LGBM_CV_PARAMS["n_jobs"] = _LGBM_INNER_THREADS
        _LGBM_CV_PARAMS["num_threads"] = _LGBM_INNER_THREADS
        print(
            f"Horizons: {len(horizon_indices)} of {n_h} | k grid: {k_grid} | folds: {len(fold_indices)} | "
            f"total tasks: {n_tasks} | outer threads: {N_PARALLEL_JOBS} | lgbm inner threads: {_LGBM_INNER_THREADS}",
            flush=True,
        )

        # Per-horizon worker — sorts X by MI once, then sweeps folds × k_grid in-process.
        # Optimisations vs prior layout:
        #   1. argsort + X reorder happens once per HORIZON (was: once per (h, fold)). Saves N_CV_SPLITS×
        #      sort/copy work per horizon. Critical because X_sorted is the dominant memory traffic.
        #   2. Fold-level fancy-index slicing happens once per fold (was: once per (h, fold)).
        #   3. Finite-mask computed once per fold; column slice X[:, :k] is zero-copy on F-order arrays.
        #   4. Uses lgb.train + lgb.Dataset directly (skips sklearn wrapper overhead, ~5-15% faster).
        #   5. No eval_set / no early stopping — fixed n_estimators=200 (see _LGBM_CV_PARAMS).
        # backend="loky": LightGBM's C++ Dataset/train are NOT thread-safe on Windows (concurrent calls
        # race in the Dataset allocator and segfault with "access violation in LGBM_DatasetFree").
        # Process workers each get their own C++ state. Joblib memmaps X_arr/Y_arr/mi_arr (>1MB) so
        # workers share the underlying buffers zero-copy on Linux/macOS, and copy on Windows.
        def _eval_horizon(h_idx, X_arr, Y_arr, mi_arr, fold_indices, k_grid, train_params, num_boost_round):
            import os
            # Force single-threaded BLAS/OMP/MKL inside the worker BEFORE numpy/lightgbm do any
            # parallel work. The parent sets these to _NCPU (47) for its own data prep, but workers
            # inherit that env → with N_PARALLEL_JOBS process workers each spawning ~_NCPU threads
            # we get _NCPU² thread contention that pegs the box without making progress.
            # threadpoolctl pins already-loaded BLAS pools too (env vars only affect new pools).
            for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                       "NUMEXPR_NUM_THREADS", "NUMEXPR_MAX_THREADS"):
                os.environ[_v] = "1"
            try:
                from threadpoolctl import threadpool_limits
                threadpool_limits(1)
            except Exception:
                pass
            import warnings
            import numpy as np
            import lightgbm as lgb
            import time as _time
            warnings.filterwarnings("ignore", message="X does not have valid feature names", category=UserWarning)
            _t_h = _time.perf_counter()

            # Sort feature columns by MI for this horizon — done once, reused for every fold and every k.
            order_h = np.argsort(mi_arr[:, h_idx])[::-1]
            X_sorted = np.asfortranarray(X_arr[:, order_h])  # one (n_rows, n_features) copy per horizon
            y = Y_arr[:, h_idx]

            per_k_fold_maes = {k: [] for k in k_grid}

            for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
                # Slice train/val rows once — these are the expensive fancy-index copies.
                X_tr_full = X_sorted[train_idx]
                X_va_full = X_sorted[val_idx]
                y_tr, y_va = y[train_idx], y[val_idx]
                mask_tr = np.isfinite(y_tr)
                mask_va = np.isfinite(y_va)
                if mask_tr.sum() < 50 or mask_va.sum() < 10:
                    continue

                X_tr_masked = X_tr_full[mask_tr]
                X_va_masked = X_va_full[mask_va]
                y_tr_masked = y_tr[mask_tr]
                y_va_masked = y_va[mask_va]

                for k in k_grid:
                    # Zero-copy column views on F-order arrays.
                    X_tr_k = X_tr_masked[:, :k]
                    X_va_k = X_va_masked[:, :k]
                    # lgb.Dataset built per (fold, k) — bin construction is unavoidable when columns change,
                    # but skipping the sklearn wrapper still saves measurable overhead per fit.
                    dtrain = lgb.Dataset(X_tr_k, label=y_tr_masked, free_raw_data=False)
                    booster = lgb.train(train_params, dtrain, num_boost_round=num_boost_round)
                    preds = booster.predict(X_va_k)
                    mae = float(np.mean(np.abs(y_va_masked - preds)))
                    per_k_fold_maes[k].append(mae)
                    del booster, dtrain
                del X_tr_full, X_va_full, X_tr_masked, X_va_masked

            del X_sorted
            return h_idx, per_k_fold_maes, _time.perf_counter() - _t_h

        tasks = list(horizon_indices)

        # Strip eval_set-only params from the dict before passing to lgb.train.
        _train_params = {k: v for k, v in _LGBM_CV_PARAMS.items() if k != "n_estimators"}
        _num_boost_round = _LGBM_CV_PARAMS["n_estimators"]

        # Warm up workers: dispatch trivial no-ops so the loky pool spawns,
        # forks, and imports lightgbm BEFORE the real workload starts. Without
        # this you stare at a 0/N bar for 30-60s while workers cold-start.
        # Dispatch 2x N_PARALLEL_JOBS so every worker is guaranteed at least one task.
        def _warmup(_):
            import os, lightgbm as lgb  # noqa: F401  (force import in worker)
            return os.getpid()
        print(f"  warming up {N_PARALLEL_JOBS} workers (forking + lightgbm import)...", flush=True)
        _t_warm = _time.perf_counter()
        _warm_n = N_PARALLEL_JOBS * 2
        _warm_gen = Parallel(n_jobs=N_PARALLEL_JOBS, backend="loky", batch_size=1, return_as="generator_unordered")(
            delayed(_warmup)(i) for i in range(_warm_n)
        )
        _warm_pids = set()
        for _pid in tqdm(_warm_gen, total=_warm_n, desc=f"warmup seed {_seed_idx + 1}/{len(_SUBSAMPLE_SEEDS)}", unit="worker", smoothing=0.05, mininterval=0.2, dynamic_ncols=True, leave=True):
            _warm_pids.add(_pid)
        print(f"    warmup done in {_time.perf_counter() - _t_warm:.1f}s ({len(_warm_pids)} unique worker PIDs)", flush=True)
        print(f"  dispatching {len(tasks)} CV tasks across {N_PARALLEL_JOBS} workers (first results may take a few seconds)", flush=True)
        try:
            _t_dispatch = _time.perf_counter()
            # backend="loky" → process workers (LightGBM C++ is not thread-safe on Windows).
            # max_nbytes="1M" → joblib memmaps any arg larger than 1MB so X_arr/Y_arr/mi_arr are shared
            # via the OS page cache rather than pickled per task.
            results_gen = Parallel(
                n_jobs=N_PARALLEL_JOBS,
                backend="loky",
                batch_size=1,
                max_nbytes="1M",
                return_as="generator_unordered",
            )(
                delayed(_eval_horizon)(h_idx, X_arr, Y_arr, mi_arr, fold_indices, k_grid, _train_params, _num_boost_round)
                for h_idx in tasks
            )

            _first = True
            _pbar = tqdm(
                results_gen,
                total=len(tasks),
                desc=f"CV seed {_seed_idx + 1}/{len(_SUBSAMPLE_SEEDS)}",
                unit="horizon",
                smoothing=0.3,
            )
            for h_idx, per_k_fold_maes, h_secs in _pbar:
                if _first:
                    _pbar.write(f"    first CV result received after {_time.perf_counter() - _t_dispatch:.1f}s")
                    _first = False
                # Append (don't overwrite) so MAEs from prior seeds are preserved and averaged later.
                for k, maes in per_k_fold_maes.items():
                    fold_mae_acc.setdefault((h_idx, k), []).extend(maes)
                _pbar.set_postfix({"h": h_idx, "last_s": f"{h_secs:.1f}"}, refresh=False)

        finally:
            # Force-shutdown loky pool so workers cannot orphan if the
            # kernel later crashes / is restarted mid-cell.
            try:
                get_reusable_executor().shutdown(wait=True, kill_workers=True)
            except Exception:
                pass
        # Free the per-seed arrays before the next iteration to avoid 2× peak memory.
        del X_arr, Y_arr, mi_arr, fold_indices

    cv_records = [
        {"horizon": horizon_cols[h_idx], "k": k, "cv_mae": float(np.mean(maes)) if maes else float("inf"), "n_samples": len(maes)}
        for (h_idx, k), maes in fold_mae_acc.items()
    ]

    cv_df = pd.DataFrame(cv_records)
    best_idx = cv_df.groupby("horizon")["cv_mae"].idxmin()
    horizon_best_k = cv_df.loc[best_idx].rename(columns={"k": "best_k"}).reset_index(drop=True)

    display(horizon_best_k)
    return horizon_best_k

horizon_best_k = feature_data_selection(
    feature_data_unique,
    features,
    future_prediction_targets_agg,
    subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_START"]),
    subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_END"]),
    subsample_amount=int(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_AMOUNT"]),
)
_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_out_path = resolve(f"3_Features_select/Selected_features/{_target}_horizon_best_k_{_stem}.parquet")
print(f"Writing horizon_best_k parquet: {_out_path}", flush=True)
_t = _time.perf_counter()
horizon_best_k.to_parquet(_out_path)
print(f"  write done in {_time.perf_counter() - _t:.1f}s", flush=True)


Loading unique feature_data parquet: /home/ec2-user/Forecasting/3_Features_select/../3_Features_select/Selected_features/NSW_feature_data_unique_1_dispatch_price.parquet
  loaded feature_data_unique: shape=(440, 102) in 0.0s
Loading features parquet: /home/ec2-user/Forecasting/3_Features_select/../2_Features_build/Feature_data/1_dispatch_price.parquet
  loaded features: shape=(736417, 634) in 1.4s
Loading aggregated targets parquet: /home/ec2-user/Forecasting/3_Features_select/../3_Features_select/Selected_features/NSW_targets_agg_1_dispatch_price.parquet
  loaded targets_agg: shape=(736417, 96) in 0.2s
  available features: 440 of 440 (intersection with loaded features parquet)

=== seed 1/5 (rng=42) ===
CV subsample: 200,000 rows (of 525,889 in range 2019-01-01 – 2024-01-01)
  materialising X_arr / Y_arr / mi_arr as float32 ndarrays
    arrays ready in 0.8s | X=336MB Y=73MB mi=0MB
X_arr: (200000, 440) (352 MB) | threads: 47
  worker mem budget: 2686MB/worker, 58.5GB available → mem-c

warmup seed 1/5: 100%|██████████| 44/44 [00:02<00:00, 21.07worker/s]

    warmup done in 2.1s (22 unique worker PIDs)
  dispatching 96 CV tasks across 22 workers (first results may take a few seconds)



CV seed 1/5:   0%|          | 0/96 [02:20<?, ?horizon/s]


KeyboardInterrupt: 

Gain-based re-ranking — refit one LightGBM at best_k per horizon on the full subsample, extract gain importance, persist gain matrix for stage 6.

In [ ]:
def feature_gain_ranking(
    feature_data,
    features,
    future_prediction_targets_agg,
    horizon_best_k,
    subsample_start: pd.Timestamp,
    subsample_end: pd.Timestamp,
    subsample_amount: int,
):
    # Mirrors the LightGBM CV config in feature_data_selection() so the gain ranking is
    # consistent with the k that CV picked. No folds, no early stopping — one fit per horizon
    # on the full subsample using the top-best_k MI features for that horizon.
    _LGBM_GAIN_PARAMS = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": 200,
        "learning_rate": 0.1,
        "num_leaves": 31,
        "min_child_samples": 20,
        "bagging_fraction": 0.5,
        "bagging_freq": 1,
        "force_col_wise": True,
        "random_state": 42,
        "n_jobs": _NCPU,
        "num_threads": _NCPU,
        "verbose": -1,
    }

    horizon_cols = [c for c in feature_data.columns if c.startswith("h") and c[1:].isdigit()]
    mi_matrix = feature_data.set_index("feature")[horizon_cols]
    available_features = [f for f in mi_matrix.index if f in features.columns]
    print(f"  available features for gain ranking: {len(available_features)}", flush=True)

    # Same subsample logic & seed as feature_data_selection so the underlying rows match the CV.
    shared_idx = features.index.intersection(future_prediction_targets_agg.index)
    shared_idx = shared_idx[(shared_idx >= subsample_start) & (shared_idx <= subsample_end)]
    n_total = len(shared_idx)
    n_rows = min(subsample_amount, n_total)
    if n_rows < n_total:
        rng = np.random.default_rng(42)
        chosen = np.sort(rng.choice(n_total, size=n_rows, replace=False))
        shared_idx = shared_idx[chosen]
    print(f"Gain subsample: {n_rows:,} rows (of {n_total:,} in range {subsample_start.date()} – {subsample_end.date()})", flush=True)

    _t = _time.perf_counter()
    X_df = features.loc[shared_idx, available_features]
    Y_arr = future_prediction_targets_agg.loc[shared_idx].to_numpy(dtype=np.float32, copy=False)
    print(f"    arrays ready in {_time.perf_counter() - _t:.1f}s | X={X_df.values.nbytes / 1024**2:.0f}MB Y={Y_arr.nbytes / 1024**2:.0f}MB", flush=True)

    # Output: gain_matrix shaped (n_features, n_horizons) — NaN where the feature was not
    # in the top-best_k MI shortlist for that horizon (i.e. not seen by the model).
    gain_matrix = pd.DataFrame(
        np.nan,
        index=pd.Index(available_features, name="feature"),
        columns=horizon_cols,
        dtype=np.float32,
    )

    # Index horizon_best_k for fast lookup
    best_k_lookup = horizon_best_k.set_index("horizon")["best_k"].to_dict()

    train_params = {k: v for k, v in _LGBM_GAIN_PARAMS.items() if k != "n_estimators"}
    num_boost_round = _LGBM_GAIN_PARAMS["n_estimators"]

    for h in tqdm(horizon_cols, desc="Gain ranking"):
        if h not in best_k_lookup or h not in mi_matrix.columns:
            continue
        k = int(best_k_lookup[h])
        if k <= 0:
            continue
        # Top-best_k features by MI for this horizon (matches stage 6's MI-based shortlist).
        top_k_features = mi_matrix[h].nlargest(k).index.tolist()
        top_k_features = [f for f in top_k_features if f in X_df.columns]
        if not top_k_features:
            continue

        y = Y_arr[:, horizon_cols.index(h)]
        mask = np.isfinite(y)
        if mask.sum() < 50:
            continue

        X_h = X_df[top_k_features].to_numpy(dtype=np.float32, copy=False)[mask]
        y_h = y[mask]

        dtrain = lgb.Dataset(X_h, label=y_h, free_raw_data=False)
        booster = lgb.train(train_params, dtrain, num_boost_round=num_boost_round)
        gains = booster.feature_importance(importance_type="gain")
        gain_matrix.loc[top_k_features, h] = gains.astype(np.float32)
        del booster, dtrain, X_h

    n_h_filled = gain_matrix.notna().any(axis=0).sum()
    n_f_used = gain_matrix.notna().any(axis=1).sum()
    print(f"  gain matrix filled: {n_h_filled} horizons, {n_f_used} unique features touched", flush=True)
    return gain_matrix.reset_index()

gain_matrix = feature_gain_ranking(
    feature_data_unique,
    features,
    future_prediction_targets_agg,
    horizon_best_k,
    subsample_start=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_START"]),
    subsample_end=pd.Timestamp(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_END"]),
    subsample_amount=int(os.environ["FEATURE_SELECTION_CV_SUBSAMPLE_AMOUNT"]),
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
_gain_path = resolve(f"3_Features_select/Selected_features/{_target}_gain_matrix_{_stem}.parquet")
print(f"Writing gain_matrix parquet: {_gain_path}", flush=True)
_t = _time.perf_counter()
gain_matrix.to_parquet(_gain_path)
print(f"  write done in {_time.perf_counter() - _t:.1f}s", flush=True)


  available features for gain ranking: 440
Gain subsample: 200,000 rows (of 525,889 in range 2019-01-01 – 2024-01-01)


    arrays ready in 0.6s | X=336MB Y=73MB


Gain ranking: 100%|██████████| 96/96 [00:34<00:00,  2.82it/s]

  gain matrix filled: 96 horizons, 85 unique features touched
Writing gain_matrix parquet: /home/ec2-user/Forecasting/3_Features_select/../3_Features_select/Selected_features/NSW_gain_matrix_1_dispatch_price.parquet


  write done in 0.2s


In [ ]:
%reset -f